In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Customer_Churn.csv")

df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

C:\Users\Mansi vaishya\AppData\Local\Temp\ipykernel_31260\1597580473.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [2]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [3]:
df["Monthly_to_Total"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)
df["Tenure_Group"] = pd.cut(df["tenure"],
                            bins=[0, 12, 24, 48, 72],
                            labels=["0-1yr", "1-2yr", "2-4yr", "4-6yr"])

In [4]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

In [6]:
df["Tenure_Group"] = df["Tenure_Group"].fillna(df["Tenure_Group"].mode()[0])

In [7]:
df = pd.get_dummies(df, drop_first=True)

In [8]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [11]:
df.Churn.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [12]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


X = df.drop("Churn", axis=1)
y = df["Churn"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "Monthly_to_Total"]
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])


smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)


param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 12, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [2, 4],
    'class_weight': ['balanced']
}


rf_base = RandomForestClassifier(random_state=42)


grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    cv=3, 
    scoring='f1', 
    n_jobs=-1,     
    verbose=2
)


print("Starting Hyperparameter Tuning...")
grid_search.fit(X_train_res, y_train_res)


print("\nBest Parameters Found:")
print(grid_search.best_params_)


best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\nFinal Classification Report with Tuned Model:")
print(classification_report(y_test, y_pred))

Starting Hyperparameter Tuning...
Fitting 3 folds for each of 54 candidates, totalling 162 fits

Best Parameters Found:
{'class_weight': 'balanced', 'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 300}

Final Classification Report with Tuned Model:
              precision    recall  f1-score   support

           0       0.89      0.80      0.84      1036
           1       0.57      0.72      0.64       373

    accuracy                           0.78      1409
   macro avg       0.73      0.76      0.74      1409
weighted avg       0.80      0.78      0.79      1409



In [13]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=12,
                       min_samples_leaf=2, min_samples_split=5,
                       n_estimators=300, random_state=42)

In [14]:
X_train_res.shape, y_train_res.shape

((8276, 23), (8276,))

In [15]:
model.score(X_train_res, y_train_res)

0.8691396810053166

In [16]:
import pandas as pd

feature_imp = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feature_imp.head(15))

               Feature  Importance
19    Monthly_to_Total    0.142141
14            Contract    0.133396
17      MonthlyCharges    0.119461
18        TotalCharges    0.103934
4               tenure    0.103682
8       OnlineSecurity    0.068240
11         TechSupport    0.051191
7      InternetService    0.040174
16       PaymentMethod    0.035730
9         OnlineBackup    0.026384
22  Tenure_Group_4-6yr    0.021223
15    PaperlessBilling    0.020782
10    DeviceProtection    0.019259
6        MultipleLines    0.016048
0               gender    0.015057


In [17]:
model.score(X_train, y_train)

0.9011359602413915

In [18]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

# print(classification_report(y_test, y_pred))

In [19]:
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [20]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7920511000709723


In [21]:
precision_score(y_test, y_pred), recall_score(y_test, y_pred), f1_score(y_test, y_pred)

(0.5921658986175116, 0.6890080428954424, 0.6369268897149938)